# Bygga med Meta-familjens modeller 

## Introduktion 

Denna lektion kommer att täcka: 

- Utforska de två huvudmodellerna i Meta-familjen - Llama 3.1 och Llama 3.2 
- Förstå användningsfall och scenarier för varje modell 
- Kodexempel för att visa varje modells unika funktioner 


## Meta-familjen av modeller 

I denna lektion kommer vi att utforska 2 modeller från Meta-familjen eller "Llama Herd" - Llama 3.1 och Llama 3.2 

Dessa modeller finns i olika varianter och är tillgängliga i [Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Notera:** GitHub Models avvecklas i slutet av juli 2026. Här finns mer information om att använda [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) för att prototypa med AI-modeller.

Modellvarianter: 
- Llama 3.1 - 70B Instruct 
- Llama 3.1 - 405B Instruct 
- Llama 3.2 - 11B Vision Instruct 
- Llama 3.2 - 90B Vision Instruct 

*Notera: Llama 3 finns också i Microsoft Foundry Models men kommer inte att täckas i denna lektion*



## Llama 3.1 

Med 405 miljarder parametrar passar Llama 3.1 in i kategorin öppen källkod LLM. 

Modellen är en uppgradering av den tidigare utgåvan Llama 3 genom att erbjuda: 

- Större kontextfönster - 128k token kontra 8k token 
- Större max antal utgående token - 4096 kontra 2048 
- Bättre flerspråkigt stöd - tack vare ökat antal traintoken 

Detta gör att Llama 3.1 kan hantera mer komplexa användningsfall vid skapande av GenAI-applikationer, inklusive: 
- Inbyggd funktionsanrop - förmågan att kalla på externa verktyg och funktioner utanför LLM-arbetsflödet
- Bättre RAG-prestanda - tack vare högre kontextfönster 
- Syntetisk datagenerering - förmågan att skapa effektiv data för uppgifter som finjustering 



### Inbyggd funktionsanrop 

Llama 3.1 har förfinats för att vara mer effektiv på att göra funktions- eller verktygsanrop. Den har också två inbyggda verktyg som modellen kan identifiera att den behöver använda baserat på användarens prompt. Dessa verktyg är: 

- **Brave Search** - Kan användas för att få aktuell information som vädret genom att utföra en webbsökning 
- **Wolfram Alpha** - Kan användas för mer komplexa matematiska beräkningar så att du inte behöver skriva egna funktioner. 

Du kan också skapa dina egna anpassade verktyg som LLM kan anropa. 

I kodexemplet nedan: 

- Definierar vi de tillgängliga verktygen (brave_search, wolfram_alpha) i systemprompten. 
- Skicka en användarprompt som frågar om vädret i en viss stad. 
- LLM kommer att svara med ett verktygsanrop till Brave Search-verktyget som kommer att se ut så här `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Notera: Detta exempel gör endast verktygsanropet, om du vill få resultaten måste du skapa ett gratis konto på Brave API-sidan och definiera funktionen själv` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Trots att det är en LLM har Llama 3.1 en begränsning vad gäller multimodalitet. Det vill säga, att kunna använda olika typer av indata såsom bilder som prompts och ge svar. Denna förmåga är en av huvudfunktionerna i Llama 3.2. Dessa funktioner inkluderar också: 

- Multimodalitet - har förmågan att utvärdera både text- och bildprompter 
- Små till medelstora storleksvarianter (11B och 90B) - detta ger flexibla distributionsalternativ, 
- Endast text-varianter (1B och 3B) - detta tillåter modellen att distribueras på edge / mobila enheter och ger låg latens 

Stödet för multimodalitet representerar ett stort steg inom öppna källkodsmodeller. Kodexemplet nedan tar både en bild- och textprompt för att få en analys av bilden från Llama 3.2 90B. 

### Multimodalt stöd med Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Lärandet slutar inte här, fortsätt resan

Efter att du har genomfört den här lektionen, kolla in vår [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) för att fortsätta utveckla dina kunskaper inom Generativ AI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
